# Chapter 6 — Regularisation · *PyTorch Companion*
**Track:** ML from Scratch · California Housing dataset

> This is the PyTorch companion to [notebook.ipynb](notebook.ipynb). All theoretical content
> and sklearn experiments are preserved verbatim. Wherever the original uses TensorFlow/Keras,
> a runnable **PyTorch equivalent** is added directly below.

## Core Idea

Regularisation closes the gap between training performance and test performance.

| Tool | Mechanism |
|---|---|
| L2 (Weight Decay) | penalise large weights — shrink all toward zero |
| L1 (Lasso) | penalise weight magnitude — push many to exactly zero |
| Dropout | randomly zero neurons during training — force redundancy |
| Early Stopping | halt when validation loss stops improving |

In [ ]:
# TODO: Implement this cell
# Hint:
#   import numpy as np
#   import matplotlib.pyplot as plt
#   from sklearn.datasets import fetch_california_housing
#   from sklearn.model_selection import train_test_split


## Baseline — No Regularisation

Establish the overfitting reference point. We expect train R² >> test R².

In [ ]:
# TODO: Implement this cell
# Hint:
#   base = MLPRegressor(
#       hidden_layer_sizes=(128, 64),
#       activation='relu',
#       solver='adam',


## L2 Regularisation — Math

$$\mathcal{L}_\text{L2} = \mathcal{L}_\text{MSE} + \lambda \sum_l \|\mathbf{W}_l\|_F^2$$

Gradient: $\;\nabla_W \mathcal{L}_\text{L2} = \nabla_W \mathcal{L}_\text{MSE} + 2\lambda \mathbf{W}$

Update: $\;\mathbf{W} \leftarrow (1 - 2\eta\lambda)\mathbf{W} - \eta \nabla_W \mathcal{L}_\text{MSE}$

The factor $(1-2\eta\lambda) < 1$ **decays** the weight every step — hence "weight decay".

In [ ]:
# TODO: Implement this cell
# Hint:
#   alphas = [0.0, 1e-5, 1e-4, 1e-3, 1e-2, 0.1]
#   l2_results = []
#   for alpha in alphas:
#       m = MLPRegressor(


## L1 vs L2 — Sparsity Comparison

L1 pushes weights to exactly zero (sparse); L2 shrinks them but almost never reaches exactly zero.

We demo this on a single linear layer (sklearn Lasso vs Ridge) to isolate the effect before applying it to a neural net.

In [ ]:
# TODO: Implement this cell
# Hint:
#   from sklearn.linear_model import Lasso, Ridge
#   alpha_val = 0.05
#   lasso = Lasso(alpha=alpha_val).fit(X_tr_s, y_tr)
#   ridge = Ridge(alpha=alpha_val).fit(X_tr_s, y_tr)


## Dropout — Manual Inverted Dropout

$$\tilde{\mathbf{h}} = \frac{1}{1-p} \cdot (\mathbf{h} \odot \mathbf{m}), \quad m_i \sim \text{Bernoulli}(1-p)$$

Scaling by $\frac{1}{1-p}$ keeps the **expected activation magnitude** the same as without dropout, so test inference (where dropout is disabled) sees the same scale it was trained with.

In [ ]:
# TODO: Implement this cell
# Hint:
#   rng = np.random.default_rng(42)
#   def dropout(h, p, training=True, rng=None):
#       """Inverted dropout.
#       h        : activation tensor, shape (n_samples, n_units)


In [ ]:
# TODO: Implement this cell
# Hint:
#   h_sample = rng.normal(0, 1, (1, 64))   # one sample, 64 neurons
#   h_after  = dropout(h_sample, p=0.4, training=True, rng=rng)
#   fig, axes = plt.subplots(2, 1, figsize=(14, 3), sharex=True)
#   axes[0].bar(range(64), h_sample[0], color='steelblue', alpha=0.8)


## Early Stopping

Train a network with early stopping enabled. Track training and validation loss per epoch to see the overfitting inflection point.

In [ ]:
# TODO: Implement this cell
# Hint:
#   es_model = MLPRegressor(
#       hidden_layer_sizes=(128, 64),
#       activation='relu',
#       solver='adam',


## Full Comparison — All Regularisation Strategies

In [ ]:
# TODO: Implement this cell
# Hint:
#   configs = [
#       ('Baseline',              dict(alpha=0.0,  early_stopping=False, max_iter...
#       ('L2 (α=1e-4)',           dict(alpha=1e-4, early_stopping=False, max_iter...
#       ('L2 (α=1e-3)',           dict(alpha=1e-3, early_stopping=False, max_iter...


## Trap 1 — Dropout on the Output Layer

sklearn's `MLPRegressor` doesn't expose dropout directly, so we demonstrate with the manual forward pass from Ch.5.

We show that applying dropout to the **output** introduces random noise into predictions — the mean prediction is preserved but individual predictions are wildly off.

In [ ]:
# TODO: Implement this cell
# Hint:
#   rng_trap = np.random.default_rng(42)
#   def relu(z): return np.maximum(0, z)
#   def he_init(n_in, n_out):
#       return rng_trap.normal(0, np.sqrt(2 / n_in), (n_in, n_out))


## Trap 2 — Validation Contamination (Scaler Leak)

Fitting `StandardScaler` on train + val leaks test-set statistics into the model, giving an artificially optimistic validation score.

In [ ]:
# TODO: Implement this cell
# Hint:
#   scaler_leak = StandardScaler()
#   X_all_s_leak = scaler_leak.fit_transform(X)   # using all 20640 rows!
#   scaler_ok   = StandardScaler().fit(X_tr)
#   _, X_val_indices = train_test_split(


## Exercises

**Exercise 1 — L2 + Dropout combination**
sklearn's `MLPRegressor` doesn't support dropout natively. Use `tensorflow.keras` to build a network with `Dropout(0.3)` after each hidden layer and `kernel_regularizer=l2(1e-4)`. Compare test R² to the sklearn L2 model.

**Exercise 2 — Patience sensitivity**
Run early stopping with `n_iter_no_change` in `[5, 10, 20, 50]`. Plot the epoch at which each stops and its test R². Is there a point of diminishing returns?

**Exercise 3 — Weight distribution under L2**
After fitting models with `alpha` in `[0, 1e-4, 1e-2]`, access `model.coefs_[0]` (first-layer weights). Plot histograms of the weight distributions side by side. How does increasing α change the shape and spread?

In [ ]:
# TODO: Implement this cell
# Hint:
#   # see solution cell for the required API calls


### PyTorch equivalent — Exercise 1 (Dropout + L2)

Key API mappings vs the Keras scaffold above:

| Keras | PyTorch |
|---|---|
| `layers.Dense(n, activation='relu')` | `nn.Linear(in, n)` + `nn.ReLU()` (or `F.relu` in `forward`) |
| `layers.Dropout(0.3)` | `nn.Dropout(0.3)` — **only active in `model.train()` mode** |
| `kernel_regularizer=l2(1e-4)` | `weight_decay=2e-4` on the optimizer (note: Keras `l2(λ)` adds `λ·‖W‖²`, PyTorch `weight_decay=wd` adds `wd/2·‖W‖²`, so `wd = 2λ` for an exact match) |
| `model.compile(optimizer='adam', loss='mse')` | `optim.Adam(...)` + `nn.MSELoss()` |
| `model.fit(..., validation_data=...)` | explicit loop: zero grad → forward → loss → backward → step |
| Dropout auto-disabled at inference | **must call `model.eval()`** before `predict` |

In [ ]:
# TODO: Implement this cell
# Hint:
#   import torch
#   import torch.nn as nn
#   import torch.optim as optim
#   from torch.utils.data import DataLoader, TensorDataset


In [ ]:
# TODO: Implement this cell
# Hint:
#   patience_values = [5, 10, 20, 50]
#   patience_results = []
#   for p in patience_values:
#       m = MLPRegressor(


In [ ]:
# TODO: Implement this cell
# Hint:
#   alphas_dist = [0, 1e-4, 1e-2]
#   fig, axes = plt.subplots(1, 3, figsize=(13, 3))
#   for ax, alpha in zip(axes, alphas_dist):
#       m = MLPRegressor(
